# Evaluate complete conversations, not isolated turns

This credential-free lab adapts MLflow's [multi-turn agent cookbook](https://mlflow.org/cookbook/multi-turn-agent/) to the platform's session and release controls. Single-turn scores miss failures that emerge across a conversation: an unanswered follow-up, unresolved frustration, or policy drift after repeated requests.

Production state must live in the application or a durable framework store. A process-global conversation dictionary is acceptable only as a toy fixture and is not used here. Each real turn gets its own trace and shares one opaque, pseudonymous session ID.

In [ ]:
import sys
from pathlib import Path

repo_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "examples" / "support").is_dir()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open the cloned repository as your workspace.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## 1. Review three synthetic sessions

The fixtures are deliberately small and deterministic. Every session is scoped to the same application release, environment, and evaluation batch so stale traces from another run cannot contaminate the result.

In [ ]:
from examples.support.agent_assurance import EVAL_BATCH, multi_turn_sessions

SESSIONS = multi_turn_sessions()
assert len({session["session_id"] for session in SESSIONS}) == len(SESSIONS)
assert all(session["eval_batch"] == EVAL_BATCH for session in SESSIONS)

## 2. Score each turn in isolation -- everything passes

A turn-level gate sees one user message and one assistant reply. Every reply below is
on-topic and free of prohibited investment advice, so a per-turn policy check passes
every row. Nothing visible in a single turn reveals that the same sentence was already
given, that an explicit escalation request went unanswered, or that half of a compound
question was dropped. That blindness is the reason this lab evaluates sessions.

In [ ]:
def turn_passes(reply: str) -> bool:
    text = reply.casefold()
    prohibited = ("buy the stock", "sell the stock", "buy shares")
    return bool(text.strip()) and not any(phrase in text for phrase in prohibited)


turn_rows = [
    {
        "session_id": session["session_id"],
        "turn": index // 2 + 1,
        "reply": turns[index + 1]["content"],
        "turn_passed": turn_passes(turns[index + 1]["content"]),
    }
    for session in SESSIONS
    for turns in [session["turns"]]
    for index in range(0, len(turns) - 1, 2)
]
turn_pass_rate = sum(row["turn_passed"] for row in turn_rows) / len(turn_rows)
assert turn_pass_rate == 1.0
print(
    f"turn-level checks: {len(turn_rows)}/{len(turn_rows)} replies pass "
    "-- the per-turn view sees no problem"
)

## 3. Convert categorical session outcomes into explicit gate metrics

The connected MLflow scorers return categorical judgments. A release policy still needs named numeric metrics and critical-case rules. This offline layer makes that conversion visible and keeps deterministic policy checks independent of an LLM judge.

In [ ]:
from examples.support.agent_assurance import build_session_report

session_report = build_session_report(SESSIONS)
print(session_report.to_string(index=False))

In [ ]:
import json

from examples.support.agent_assurance import session_gate as evaluate_session_gate

session_metrics, session_policy, session_gate = evaluate_session_gate(session_report)
session_gate_summary = {
    "metrics": session_metrics,
    "gate_passed": session_gate.passed,
    "decision": "adopt" if session_gate.passed else "reject",
    "failures": [failure.model_dump(mode="json") for failure in session_gate.failures],
}
print(json.dumps(session_gate_summary, indent=2))

### Print the conversations the per-turn view waved through

The gate above rejects what every per-turn check passed. Reading the failing
transcripts shows why the session is the right unit of evaluation.

In [ ]:
failing = session_report[session_report["critical_session_pass"] == 0.0]
print(
    f"turn-level pass rate {turn_pass_rate:.2f}, yet {len(failing)} of "
    f"{len(session_report)} sessions fail at session scope"
)
indexed = session_report.set_index("session_id")
for session_id in failing["session_id"]:
    row = indexed.loc[session_id]
    session = next(s for s in SESSIONS if s["session_id"] == session_id)
    print(
        f"\n{session_id}: complete={row['conversation_complete']:.0f} "
        f"guidelines={row['conversational_guidelines']:.0f} "
        f"frustration={row['frustration']}"
    )
    for turn in session["turns"]:
        print(f"  {turn['role']:>9}: {turn['content']}")

## 4. Optional connected MLflow conversational judges

The native `ConversationCompleteness`, `ConversationalGuidelines`, and `UserFrustration` scorers are experimental LLM judges. They evaluate **pre-collected traces** and do not accept a `predict_fn` for multi-turn evaluation.

For each real turn, open `mlflow.tracing.context(session_id=opaque_session_id)`, create exactly one traced agent invocation, and tag the trace with the evaluation batch, application release, and environment. Query all three tags before scoring. Do not attach a raw personal identifier.

In [ ]:
from examples.support.agent_assurance import run_native_conversational_judges

RUN_NATIVE_CONVERSATIONAL_JUDGES = False
SOURCE_TRACE_EXPERIMENT_ID = None
JUDGE_MODEL_URI = None  # Resolve the configured logical judge-model keylessly.
if RUN_NATIVE_CONVERSATIONAL_JUDGES:
    if not SOURCE_TRACE_EXPERIMENT_ID or not JUDGE_MODEL_URI:
        raise ValueError(
            "Set the source-trace experiment ID and governed judge model URI first"
        )
    print(
        run_native_conversational_judges(
            SESSIONS,
            source_trace_experiment_id=SOURCE_TRACE_EXPERIMENT_ID,
            judge_model_uri=JUDGE_MODEL_URI,
        )
    )
else:
    print("CONNECTED CONVERSATIONAL JUDGES SKIPPED")

## Result

The fixture is rejected because aggregate success cannot excuse one unresolved critical session. The contrast is the lesson: the same assistant replies pass every turn-scoped check, and two of three conversations still fail the session gate. In a connected evaluation, preserve each judge rationale, fail on scorer errors, and keep these judges report-only until held-out human calibration meets the approved agreement threshold.